In [5]:
import re
from IPython.display import display, Markdown

from emu_renewal.inputs import DEFAULT_END_TIME, DEATHS_WEIGHT, DEATHS_START_THRESHOLD, DEFAULT_START_TIME, START_VACC_THRESHOLD_AUS, get_worldbank_national_pop
from emu_renewal.outputs import TEXT_DATE_FORMAT
from emu_renewal.run import find_run_start_time, find_run_end_time, get_deaths_target, get_cases_target, get_hosp_target

In [6]:
def get_func_blurb(function):
    docstring = function.__doc__
    blurb = re.split("(Args|Returns):", docstring)[0]
    return re.sub(r"\s+", " ", blurb)

In [16]:
txt = "### Modelled population size\n"
txt += get_func_blurb(get_worldbank_national_pop)
txt += "\n\n### Analysis period\n"
txt += get_func_blurb(find_run_start_time)
txt += f"The deaths starting threshold was set at {round(DEATHS_START_THRESHOLD * 1e6)} deaths per million population. "
txt += f"The default latest time to start the analysis was set at {DEFAULT_START_TIME.strftime(TEXT_DATE_FORMAT)}. "
txt += f"The analysis for Australia commenced once the vaccination coverage exceeded {round(START_VACC_THRESHOLD_AUS * 100)}% of its final value. \n\n"
txt += get_func_blurb(find_run_end_time)
default_end_time = DEFAULT_END_TIME.strftime(TEXT_DATE_FORMAT)
txt += "The default end date for countries other than Australia not reaching " \
    f"the vaccination threshold was set at {default_end_time}. "
txt += "\n\n### Calibration targets\n"
txt += get_func_blurb(get_deaths_target)
txt += f"The total weight for the time series of deaths was set at a value of {DEATHS_WEIGHT}.\n\n"
txt += get_func_blurb(get_cases_target)
txt += "\n\n"
txt += get_func_blurb(get_hosp_target)

In [17]:
display(Markdown(txt))

### Modelled population size
Population data were downloaded from the World Bank at [this address](https://databank.worldbank.org/source/population-estimates-and-projections#) on 01/04/2025. From this data, the population size in 2020 of country of interest was extracted. The exception was Australia, for which the population size in 2022 was used, because of its later analysis period. 

### Analysis period
For all countries except Australia, the start of the calibration period was set to be the time at which the per capita daily rate of deaths passed a specified threshold. However, if this threshold was not reached by a default start date, the simulation commenced at this default time. For Australia, the simulation commenced from the time that vaccination reached a proportion of its final value. The deaths starting threshold was set at 2 deaths per million population. The default latest time to start the analysis was set at 1 June 2020. The analysis for Australia commenced once the vaccination coverage exceeded 90% of its final value. 

For all countries but Australia, the end time for the analysis was calculated as the time that the population vaccination coverage passed a specific threshold value, provided that the vaccination coverage did reach this value by the default end time. Otherwise, a default end date is used. For Australia, the latest date for which the Google mobility data was available was used. The default end date for countries other than Australia not reaching the vaccination threshold was set at 1 December 2021. 

### Calibration targets
The number of deaths by week reported by WHO was used as the first calibration target for all countries. Any values of zero in this series were replaced with a value of 0.5 to enable comparison to modelled outputs on the log scale. Deaths was the one of two indicators for which a common dispersion parameter was used for the distribution comparison of the modelled value. The total weight for the time series of deaths was set at a value of 20.0.

The number of cases by week reported by WHO was used as the second calibration target for all countries. As for deaths, any zero values were replaced with 0.5. Cases was the second indicator for which a common dispersion parameter was applied. A target weight was applied to the series of cases such that the weight for each case observation point was the same as for each death observation. 

One hospitalisation indicator was also used for each country, where available. This indicator was the final calibration target for which a common dispersion parameter was applied. As for cases, a weight was applied to the hospitalisation series such that the weight for each observation point was the same as for each death observation. 